# Modelo 1 — construção do classificador

**O instrumento do projeto inteiro.** Ele olha um pixel de imagem de satélite e diz a que
classe de cobertura do solo ele pertence. Tudo o que o estudo de impacto afirma é contagem
sobre a saída dele.

Este notebook mostra **como ele foi feito**. Para vê-lo **funcionando**, com imagem de verdade,
use `04_demo_visual_classificador.ipynb`.

In [1]:
import json
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
m = json.loads((RAIZ / "data" / "manifests" / "dataset_v1.0.json").read_text(encoding="utf-8"))

print(f"dataset  : {m['versao']}")
print(f"linhas   : {m['n_linhas']:,}")
print(f"features : {m['n_features']}")
print(f"sites    : {len(m['sites'])}   ·   anos: {min(m['anos'])}–{max(m['anos'])}")
print(f"sensores : {', '.join(m['sensores'])}")
print(f"seed     : {m['seed']}")

dataset  : v1.0
linhas   : 3,798,750
features : 13
sites    : 16   ·   anos: 2013–2025
sensores : landsat, s2
seed     : 42


## 1. As 5 classes

Fechadas pelo time em 2026-08-27 e não renegociadas depois. Infraestrutura viária como classe
separada ficou fora do V1 — entra em "construída".

In [2]:
import yaml
cls = yaml.safe_load((RAIZ / "config" / "classes.yml").read_text(encoding="utf-8"))["classes"]
for cid, meta in sorted(cls.items()):
    if cid == 0:
        continue
    print(f"  {cid}  {meta['cor_hex']}  {meta['nome_exibicao']}")

  1  #1B5E20  Vegetação densa
  2  #8BC34A  Vegetação rala / pasto / agricultura leve
  3  #F5A623  Solo exposto / em obras
  4  #B03A2E  Área construída / urbana
  5  #1565C0  Água


## 2. De onde vêm os rótulos — e o defeito de origem

Ninguém rotulou 3,8 milhões de pixels à mão. O rótulo automático vem do **MapBiomas Coleção 9**
(anual, 2013–2023), com o **ESA WorldCover** como verificação cruzada em 2021 — o único ano de
sobreposição real entre as duas fontes.

A decisão está no `ADR-004`, e trocou a escolha original (WorldCover puro) porque a janela do
projeto virou 2013–2025: uma safra fixa aplicada a 13 anos gerava defasagem de até 8 anos, um
erro sistemático medido em **4–6% dos pixels por site**.

> **O defeito que isso carrega, e que explica o resto do notebook:** nem o MapBiomas nem o
> WorldCover têm uma classe de **canteiro de obras**. A classe 3 é rotulada por um proxy de solo
> nu natural. Por isso a rotulagem manual complementar é obrigatória — e por isso a classe 3 é a
> mais fraca do modelo.

In [3]:
print(f"fonte do rótulo: {m['fonte_label']}\n")
lm = m.get("labels_manuais", {})
if lm:
    print("rotulagem manual complementar (SV-09/SV-10):")
    for k, v in lm.items():
        if isinstance(v, dict):
            print(f"  {k}: {v}")
        else:
            print(f"  {k}: {v}")

fonte do rótulo: mapbiomas_coleção9_anual (ADR-004 opção b) + worldcover_crosscheck_2021

rotulagem manual complementar (SV-09/SV-10):
  arquivos: [{'arquivo': 'data/labels_manual/angonap-fortaleza.geojson', 'sha256': 'd37ace908bb4376df50ddfd8e28fd06d87e078ebbadef60f1390a384ff96cb17'}, {'arquivo': 'data/labels_manual/ascenty-hortolandia.geojson', 'sha256': '3158fd5eda8a4a871e6e5081b7928bf4fc78fff8d6d988154366515cdda87302'}, {'arquivo': 'data/labels_manual/ascenty-maracanau.geojson', 'sha256': '2ab4fc47509e5b357b65f2998c5bfac47c99d895474e98f2038cb5453509b6b8'}, {'arquivo': 'data/labels_manual/ascenty-osasco.geojson', 'sha256': '3b5cb1a33d6c7c45378a175c5c1ab3e0acc32c185da0d402e5ccfb75cb2fa58c'}, {'arquivo': 'data/labels_manual/ascenty-sumare.geojson', 'sha256': 'd5ac13f92fb1d727ae408819de0a296b5c7216f012515cef1ef99d22f7cc4fda'}, {'arquivo': 'data/labels_manual/ascenty-vinhedo.geojson', 'sha256': '7a50acb7bbb12566ca8a12a9ded03d44d7ab687351be9d5189735ba2ba93ac32'}, {'arquivo': 'data/labels

## 3. A amostragem — e um bug que ela escondia

Amostragem estratificada por classe, com teto por classe × site × ano × sensor.

**O teto era por contagem de pixel, e isso estava errado.** Um pixel Landsat de 30 m cobre 9× a
área de um de 10 m do Sentinel-2, então o mesmo teto significava áreas muito diferentes. As
classes abundantes enchiam o teto nos dois sensores; a classe 3 **nunca** enchia no Landsat.

Resultado medido: a classe 3 era **2,8%** das linhas Landsat contra **17,6%** das S2. E como
`sensor` é feature do modelo, ele aprendeu o prior condicionado ao sensor e o reproduzia na
saída — 2,4% previsto no Landsat contra 19,3% no S2.

A correção (teto por **área**, `ADR-006 §3`) está implementada em `sentinela.dataset` e foi
aplicada no `dataset_v2.0`. O `v1.0` documentado aqui é o de produção e ainda carrega o viés.

In [4]:
d = pd.DataFrame(m["distribuicao_classes"]).T
print("distribuição de classes no dataset_v1.0:")
print(d.to_string())
print()
am = m.get("amostragem", {})
for k, v in am.items():
    print(f"  {k}: {v}")

distribuição de classes no dataset_v1.0:
                   vegetacao_densa vegetacao_rala solo_exposto_obras construida_urbana      agua                                                                                                                            treino                                                                                                                             teste                                                                                                                                s2                                                                                                                          landsat                                                                                                                         s2_treino                                                                                                                          s2_teste                                                                                    

## 4. O split — nunca aleatório por pixel

Esta é a regra mais importante do `CLAUDE.md`, e a razão é simples: pixels vizinhos são quase
idênticos. Um split aleatório coloca o vizinho do pixel de teste dentro do treino, e a métrica
vira ficção.

São **três eixos** de separação:

| eixo | regra | o que protege |
|---|---|---|
| espacial | blocos de 1 km × 1 km inteiros | autocorrelação espacial |
| AOI (holdout) | **sites inteiros** fora do treino | generalizar para área nunca vista |
| temporal | anos inteiros | vazamento por série |

In [5]:
print(f"regra de split : {m['regra_split']}")
print(f"blocos         : {m['n_blocos']}")
print(f"\nAOIs em holdout espacial (o modelo NUNCA as viu no treino):")
for a in m["aois_holdout_espacial"]:
    print(f"  · {a}")
print(f"\nestratos que NÃO podem ser feature: {m['estratos_nao_sao_features']}")

regra de split : bloco_id = grade regular de 1km x 1km sobre coordenadas projetadas (x, y) em EPSG:31983 (nunca linha/coluna — índices de pixel significam distâncias diferentes a 10m e a 30m). Blocos inteiros (não pixels) são sorteados 70%/30% para treino/teste, com random_state=42, estratificado por AOI (site_id): cada AOI tem seu próprio sorteio 70/30 sobre a lista de blocos únicos dela, e todos os anos e sensores de um mesmo bloco vão para o mesmo split. Isso fecha os 3 vetores de vazamento de dados do projeto ao mesmo tempo: espacial (pixels vizinhos), temporal (mesmo lugar em anos consecutivos) e entre sensores (mesmo lugar, mesmo ano, duas cópias em resoluções diferentes nos anos de sobreposição 2019-2021). Regra idêntica à v0.1 (SV-11) — não foi alterada para SV-27, só aplicada a mais AOIs; a estratificação por AOI garante que nenhuma região/bioma vá inteiramente para um único split (ver `cobertura_estrato`, `regioes_sem_ambos_splits`, `biomas_sem_ambos_splits`). SV-27 acrescent

A última linha é a trava contra o erro clássico: **região, bioma e UF nunca entram como
feature.** Aprender geografia em vez de espectro funciona bem no teste e quebra na primeira AOI
nova.

## 5. O modelo

Random Forest, o baseline V1 definido pelo time. As 13 features são as 6 bandas harmonizadas
entre sensores mais os 7 índices espectrais.

In [6]:
print("as 13 features:")
for i, f in enumerate(m["lista_features"], 1):
    marca = "  ← índice calculado" if i > 6 else "  ← banda harmonizada"
    print(f"  {i:2d}. {f:<8}{marca}")

as 13 features:
   1. blue      ← banda harmonizada
   2. green     ← banda harmonizada
   3. red       ← banda harmonizada
   4. nir       ← banda harmonizada
   5. swir1     ← banda harmonizada
   6. swir2     ← banda harmonizada
   7. ndvi      ← índice calculado
   8. evi       ← índice calculado
   9. ndwi      ← índice calculado
  10. mndwi     ← índice calculado
  11. ndbi      ← índice calculado
  12. bsi       ← índice calculado
  13. ndmi      ← índice calculado


**Uma variante foi testada e decidida com número.** O treino compara duas versões — com e sem
`sensor` como feature binária — e adota a melhor. No `v1.0-tuned` a variante **com sensor**
ganhou, e o relatório registrou isso como "dependência de época residual".

Depois descobrimos que boa parte dessa dependência **era o bug do teto de amostragem**: no
`v2.0`, com o teto corrigido, a diferença entre as variantes caiu para +0,0024 e o modelo passou
a adotar a versão **sem** sensor.

## 6. A avaliação — e onde o modelo é fraco

Holdout espacial: os 3 sites que o modelo nunca viu.

In [7]:
import re
rel = (RAIZ / "reports" / "avaliacao_rf_v1.0-tuned.md").read_text(encoding="utf-8")
bloco = rel.split("## (a) Holdout espacial")[1].split("![")[0]
linhas = [l for l in bloco.splitlines() if l.strip().startswith("|")]
print("\n".join(linhas))

| classe | precision | recall | f1 | suporte |
|---|---|---|---|---|
| Vegetação densa | 0.873 | 0.885 | 0.879 | 422434 |
| Vegetação rala / pasto / agricultura leve | 0.724 | 0.748 | 0.736 | 413665 |
| Solo exposto / em obras | 0.587 | 0.572 | 0.579 | 185319 |
| Área construída / urbana | 0.792 | 0.738 | 0.764 | 436810 |
| Água | 0.894 | 0.948 | 0.920 | 235684 |
| **macro avg** | 0.774 | 0.778 | **0.776** | 1693912 |
| **weighted avg** | 0.787 | 0.788 | 0.787 | 1693912 |


**A classe 3 é a pior, e não por acaso.** F1 de 0,579 contra 0,879 da vegetação densa e 0,920 da
água. A causa está na seção 2: o rótulo dela é um proxy, porque nenhuma das fontes automáticas
tem classe de canteiro de obras.

Isso foi **confirmado depois**: o retreino com rótulos do Dynamic World — que tem uma classe
`bare` nativa — levou a classe 3 de **0,580 para 0,804**. O diagnóstico estava certo.

Esse mesmo retreino, porém, **reprovou** no critério que decide adoção (estabilidade temporal,
`ADR-006 §4`), e por isso o `rf_v1.0-tuned` continua sendo o modelo de produção. A história
completa está em `modelo-impacto-score/reports/relatorio-impacto.md`, seção 4.4.

## O que este modelo é, e o que ele não é

**É** um instrumento: entra imagem, sai classe por pixel, com um mapa de confiança junto.

**Não é** um detector de data center. Ele não sabe o que é um data center, não sabe o que é
"antes" e "depois", e não sabe que existe grupo de controle. Tudo isso é o **modelo 2**
(seleção de controle) e a **análise de impacto**, que consomem a saída dele.

**Não é perfeito, e o erro está medido:** ele troca a classe de **17,5%** dos pixels entre anos
consecutivos em terrenos onde nada mudou — 2,4× mais que o Dynamic World. Essa instabilidade é a
principal limitação conhecida do trabalho, e está documentada no relatório.